# pretraining — Python demo

Numerical companion to the entry [pretraining](https://dictionaryofml.org/terms/pretraining.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'pretraining'. A data point is one day at Krems an der Donau (GeoSphere station 3805). Its feature vector holds the hourly readings of the five previous days -- air temperature, relative humidity, air pressure, wind speed, wind direction as sine and cosine, precipitation and sunshine duration, 24 x 5 values each, d = 960 features -- and the ultimate learning task is to predict the air temperature at 15:00 (UTC) of the day, a forecast one day ahead. Its labels are kept scarce on purpose: 4 to 128 labeled days of 2024, where a linear model with 960 weights cannot be learned from the labels alone.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/pretraining.py`](https://dictionaryofml.org/terms/pretraining.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 pretraining.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "pretraining.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Pretraining a linear representation on tasks whose labels come free, then
learning a new task from a few labeled days: the 15:00 (UTC) temperature at
Krems an der Donau from five days of hourly observations.

Purpose
-------
Numerical companion to the glossary entry 'pretraining'.  A data point is
one day at Krems an der Donau (GeoSphere station 3805).  Its feature
vector holds the hourly readings of the five previous days -- air
temperature, relative humidity, air pressure, wind speed, wind direction
as sine and cosine, precipitation and sunshine duration, 24 x 5 values
each, d = 960 features -- and the ultimate learning task is to predict the
air temperature at 15:00 (UTC) of the day, a forecast one day ahead.  Its
labels are kept scarce on purpose: 4 to 128 labeled days of 2024, where a
linear model with 960 weights cannot be learned from the labels alone.

Pretraining uses the two-layer linear model of Du et al. (2021) and
Tripuraneni et al. (2021): T source tasks whose labels come free share a
d'-dimensional linear representation, y = w^T B^T x with B of size d x d'
(the k of Du et al.; the book writes a reduced dimension as d').
Here the source tasks are the T = 191 other hourly readings of the day
(every reading but the 15:00 temperature), labeled for free on all days of
2006 to 2023.  The rank-d' least squares over these tasks is reduced-rank
regression (Izenman 1975): least squares, then the rank-d' truncation of
the fitted values, then B as an orthonormal basis of the coefficient
columns.  The ultimate task then learns only d' = 4 head coefficients by
least squares on the m labeled days.  The theory promises an error that
scales with d'/m instead of d/m.

Compared on the days of 2024 that are not labeled: least squares on the
960 features from scratch (the minimum-norm solution, since m < d), ridge
regression from scratch, the pretrained representation with a least
squares head, the same with a representation pretrained only on the 48
wind-direction readings (an unrelated pretext task), and the mean of the
labeled days.  Two references: least squares on every labeled day of 2006
to 2023 (what the features allow) and persistence (yesterday's 15:00
reading).  A second experiment varies d' at m = 16 labeled days.

Nothing but numpy.  Hourly data via the GeoSphere API (klima-v2-1h, UTC
stamps), fetched once per year into pythondemos/.cache/ and read offline
afterwards.

Blocks
------
[B-data]     the hourly archive 2006-2024, days with complete windows
[B-pretrain] the rank-d' representation from the 191 free source tasks
[B-target]   the ultimate task from 4 to 128 labeled days of 2024
[B-rank]     the choice of d' at 16 labeled days
[B-plot]     the CSV files and the preview figure

Run
---
    uv run --script pythondemos/pretraining.py
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

from __future__ import annotations

import json
import urllib.request
from datetime import date, timedelta
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

OUT_DIR = Path(__file__).parent
CACHE = OUT_DIR / ".cache"
CACHE.mkdir(exist_ok=True)
report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-data]** the hourly archive 2006-2024, days with complete windows

In [ ]:
print("[B-data] hourly observations at Krems, 2006-2024")
PARAMS = ["tl", "rf", "p", "ff", "dd", "rr", "so_h"]
CHANNELS = ["tl", "rf", "p", "ff", "sin dd", "cos dd", "rr", "so_h"]
YEARS = range(2006, 2025)
WINDOW = 5                       # days of history in a feature vector
LABEL_HOUR, LABEL_CHANNEL = 15, 0   # the 15:00 (UTC) air temperature
API = ("https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1h"
       "?parameters={p}&station_ids=3805&start={y}-01-01T00:00&end={y}-12-31T23:00"
       "&output_format=geojson")


def year_data(year):
    path = CACHE / f"pretraining_hourly_{year}.json"
    if not path.exists():
        with urllib.request.urlopen(API.format(p=",".join(PARAMS), y=year), timeout=600) as resp:
            payload = json.load(resp)
        props = payload["features"][0]["properties"]["parameters"]
        json.dump({"stamps": payload["timestamps"],
                   "data": {p: props[p]["data"] for p in PARAMS}}, open(path, "w"))
    return json.load(open(path))


hours = {}                        # (date, hour) -> readings or None
for y in YEARS:
    d = year_data(y)
    for i, st in enumerate(d["stamps"]):
        vals = [d["data"][p][i] for p in PARAMS]
        hours[(st[:10], int(st[11:13]))] = None if any(v is None for v in vals) else vals


def day_block(day):
    """24 x 8 array of the readings of a complete day (dd as sine and cosine), else None."""
    rows = []
    for h in range(24):
        v = hours.get((day, h))
        if v is None:
            return None
        tl, rf, p, ff, dd, rr, so = v
        ang = np.deg2rad(dd)
        rows.append([tl, rf, p, ff, np.sin(ang), np.cos(ang), rr, so])
    return np.array(rows)


days = sorted({k[0] for k in hours})
blocks = {d: day_block(d) for d in days}
print(f"    {len(days)} days in the archive, {sum(b is not None for b in blocks.values())} with all 24 hours of all parameters")
check("[B-data] 2024-01-01 reads 4.0 at 06:00 and 8.0 at 15:00 (UTC)",
      blocks["2024-01-01"] is not None and blocks["2024-01-01"][6, 0] == 4.0 and blocks["2024-01-01"][15, 0] == 8.0)


def datapoint(day):
    """Features: the five previous CALENDAR days; the day's own 192 readings as labels."""
    d0 = date.fromisoformat(day)
    prev = [(d0 - timedelta(days=k)).isoformat() for k in range(WINDOW, 0, -1)]
    if blocks[day] is None or any(blocks.get(p) is None for p in prev):
        return None
    return np.concatenate([blocks[p].ravel() for p in prev]), blocks[day].ravel()


records = {d: datapoint(d) for d in days if blocks[d] is not None}
records = {d: r for d, r in records.items() if r is not None}
pre_days = [d for d in records if d < "2024-01-01"]
tgt_days = [d for d in records if d >= "2024-01-01"]
X_pre = np.array([records[d][0] for d in pre_days]); X_tgt = np.array([records[d][0] for d in tgt_days])
R_pre = np.array([records[d][1] for d in pre_days]); R_tgt = np.array([records[d][1] for d in tgt_days])
mu, sd = X_pre.mean(axis=0), X_pre.std(axis=0) + 1e-9
X_pre = (X_pre - mu) / sd; X_tgt = (X_tgt - mu) / sd
label_col = LABEL_HOUR * len(CHANNELS) + LABEL_CHANNEL
y_pre, y_tgt = R_pre[:, label_col], R_tgt[:, label_col]
n_pre, n_tgt, d_feat = len(pre_days), len(tgt_days), X_pre.shape[1]
print(f"    data points: {n_pre} of 2006-2023 (pretraining), {n_tgt} of 2024; {d_feat} features; "
      f"label: the 15:00 (UTC) temperature, 2024 mean {y_tgt.mean():.1f}, std {y_tgt.std():.2f} deg C")
check("[B-data] every data point has 960 features", d_feat == 960)
check("[B-data] the label column is the 15:00 temperature", CHANNELS[LABEL_CHANNEL] == "tl" and label_col == 120)
check("[B-data] 355 days of 2024 carry a complete window and a complete day", n_tgt == 355)
check("[B-data] 6209 days of 2006-2023 do", n_pre == 6209)

**[B-pretrain]** the rank-d' representation from the 191 free source tasks

In [ ]:
print("[B-pretrain] rank-d' least squares over the free source tasks (reduced-rank regression)")
K = 4
source_cols = [c for c in range(R_pre.shape[1]) if c != label_col]            # 191 tasks
wind_cols = [h * len(CHANNELS) + c for h in range(24) for c in (4, 5)]           # 48 tasks


def standardize_tasks(R, cols):
    Y = R[:, cols]
    return (Y - Y.mean(axis=0)) / (Y.std(axis=0) + 1e-9)


def representation(X, Y, k):
    """B (d x d'), orthonormal columns: the rank-d' least squares over the tasks in Y.

    Least squares W, then the rank-d' truncation of the fitted values XW keeps
    the d' right singular directions that explain most of them (Izenman 1975);
    the column space of the truncated W is the representation."""
    W = np.linalg.lstsq(X, Y, rcond=None)[0]
    _, _, Vt = np.linalg.svd(X @ W, full_matrices=False)
    Wk = W @ (Vt[:k].T @ Vt[:k])
    U, _, _ = np.linalg.svd(Wk, full_matrices=False)
    return U[:, :k]


Y_src = standardize_tasks(R_pre, source_cols)
Y_wind = standardize_tasks(R_pre, wind_cols)
B_src = representation(X_pre, Y_src, K)
B_wind = representation(X_pre, Y_wind, K)
print(f"    {len(source_cols)} source tasks, {len(wind_cols)} wind-direction tasks, d' = {K}")
check("[B-pretrain] the source tasks are the 191 other readings of the day", len(source_cols) == 191)
check("[B-pretrain] B has orthonormal columns", np.allclose(B_src.T @ B_src, np.eye(K), atol=1e-8))
# the truncation is the rank-k minimizer among the candidates it is compared with
W_full = np.linalg.lstsq(X_pre, Y_src, rcond=None)[0]
err_full = ((X_pre @ W_full - Y_src) ** 2).mean()
F = X_pre @ B_src
W_head = np.linalg.lstsq(F, Y_src, rcond=None)[0]
err_rank = ((F @ W_head - Y_src) ** 2).mean()
rng0 = np.random.default_rng(0)
B_rand = np.linalg.qr(rng0.standard_normal((d_feat, K)))[0]
err_rand = ((X_pre @ B_rand @ np.linalg.lstsq(X_pre @ B_rand, Y_src, rcond=None)[0] - Y_src) ** 2).mean()
print(f"    average squared error over the source tasks: full least squares {err_full:.3f}, "
      f"rank-{K} representation {err_rank:.3f}, a random rank-{K} subspace {err_rand:.3f} (labels standardized)")
check("[B-pretrain] the learned rank-d' subspace beats a random one on the source tasks", err_rank < err_rand)

**[B-target]** the ultimate task from 4 to 128 labeled days of 2024

In [ ]:
print("[B-target] the 15:00 temperature from 4 to 128 labeled days of 2024")
SIZES = [4, 8, 16, 32, 64, 128]
DRAWS = 10
RIDGE = 1e3
test = np.arange(n_tgt) % 2 == 0                 # every second day of 2024 is a test day
pool = np.where(~test)[0]                        # the labeled days are drawn from the others
Xte, yte = X_tgt[test], y_tgt[test]


def rmse(pred):
    return float(np.sqrt(((pred - yte) ** 2).mean()))


def ls_head(F_tr, y_tr, F_te):
    """Least squares with an offset on the given features (minimum-norm when underdetermined)."""
    A = np.hstack([F_tr, np.ones((len(F_tr), 1))])
    c = np.linalg.lstsq(A, y_tr, rcond=None)[0]
    return np.hstack([F_te, np.ones((len(F_te), 1))]) @ c


def ridge(X_tr, y_tr, X_te, lam):
    c = np.linalg.solve(X_tr.T @ X_tr + lam * np.eye(X_tr.shape[1]), X_tr.T @ (y_tr - y_tr.mean()))
    return X_te @ c + y_tr.mean()


def labeled(m, draw):
    return np.random.default_rng(1000 + draw).choice(pool, size=m, replace=False)


ceiling = rmse(ls_head(X_pre, y_pre, Xte))
persist = rmse(np.array([blocks[(date.fromisoformat(d) - timedelta(days=1)).isoformat()][LABEL_HOUR, 0]
                         for d, t in zip(tgt_days, test) if t]))
print(f"    {test.sum()} test days; least squares on all {n_pre} labeled days of 2006-2023: RMSE {ceiling:.2f}; "
      f"persistence (the previous day's 15:00 reading): {persist:.2f} deg C")
VARIANTS = ["mean", "scratch", "ridge", "pretrained", "wind"]
curve = {k: [] for k in VARIANTS}
for m in SIZES:
    res = {k: [] for k in VARIANTS}
    for r in range(DRAWS):
        lab = labeled(m, r)
        Xtr, ytr = X_tgt[lab], y_tgt[lab]
        res["mean"].append(rmse(np.full(len(yte), ytr.mean())))
        res["scratch"].append(rmse(ls_head(Xtr, ytr, Xte)))
        res["ridge"].append(rmse(ridge(Xtr, ytr, Xte, RIDGE)))
        res["pretrained"].append(rmse(ls_head(Xtr @ B_src, ytr, Xte @ B_src)))
        res["wind"].append(rmse(ls_head(Xtr @ B_wind, ytr, Xte @ B_wind)))
    for k in VARIANTS:
        curve[k].append(float(np.mean(res[k])))
    print(f"    m = {m:3d}: " + ", ".join(f"{k} {curve[k][-1]:.2f}" for k in VARIANTS) + "  (RMSE in deg C, mean of 10 draws)")
i16, i128 = SIZES.index(16), SIZES.index(128)
check("[B-target] with 16 labels the pretrained representation beats least squares from scratch with 128",
      curve["pretrained"][i16] < curve["scratch"][i128])
check("[B-target] with 16 labels the pretrained representation beats ridge regression from scratch with 128",
      curve["pretrained"][i16] < curve["ridge"][i128])
check("[B-target] from 8 labels on the pretrained representation beats the mean of the labeled days",
      all(p < b for p, b in zip(curve["pretrained"][1:], curve["mean"][1:])))
check("[B-target] with 4 labels, fewer than the d' + 1 = 5 head coefficients, it does not",
      curve["pretrained"][0] > curve["mean"][0])
check("[B-target] the wind-direction representation is worse than the full one at every m",
      all(w > p for w, p in zip(curve["wind"], curve["pretrained"])))
check("[B-target] least squares from scratch is worse than the mean below 128 labels",
      all(s > b for s, b in zip(curve["scratch"][:-1], curve["mean"][:-1])))
check("[B-target] all labels of 2006-2023 beat persistence", ceiling < persist)
print(f"    at 16 labels: pretrained {curve['pretrained'][i16]:.2f} against ridge {curve['ridge'][i16]:.2f}, "
      f"scratch {curve['scratch'][i16]:.2f}, wind {curve['wind'][i16]:.2f}, mean {curve['mean'][i16]:.2f}")

**[B-rank]** the choice of d' at 16 labeled days

In [ ]:
print("[B-rank] the dimension d' of the representation, at 16 labeled days")
RANKS = [1, 2, 4, 8, 16, 32, 64, 128]
rank_curve = []
for k in RANKS:
    Bk = representation(X_pre, Y_src, k)
    vals = []
    for r in range(DRAWS):
        lab = labeled(16, r)
        vals.append(rmse(ls_head(X_tgt[lab] @ Bk, y_tgt[lab], Xte @ Bk)))
    rank_curve.append(float(np.mean(vals)))
    print(f"    d' = {k:3d}: RMSE {rank_curve[-1]:.2f}")
best_k = RANKS[int(np.argmin(rank_curve))]
check("[B-rank] the best d' at 16 labels is below 16", best_k < 16)
check("[B-rank] d' = 128 is worse than d' = 4", rank_curve[RANKS.index(128)] > rank_curve[RANKS.index(4)])
check("[B-rank] the error peaks at d' = 16, where d' + 1 head coefficients meet 16 labels",
      RANKS[int(np.argmax(rank_curve))] == 16)

**[B-plot]** the CSV files and the preview figure

In [ ]:
print("[B-plot] CSV files and preview")
with open(OUT_DIR / "pretraining_curve.csv", "w") as fh:
    fh.write("m," + ",".join(VARIANTS) + ",ceiling,persistence\n")
    for i, m in enumerate(SIZES):
        fh.write(f"{m}," + ",".join(f"{curve[k][i]:.4f}" for k in VARIANTS) + f",{ceiling:.4f},{persist:.4f}\n")
with open(OUT_DIR / "pretraining_rank.csv", "w") as fh:
    fh.write("dprime,rmse\n")
    for k, v in zip(RANKS, rank_curve):
        fh.write(f"{k},{v:.4f}\n")
with open(OUT_DIR / "pretraining_data.csv", "w") as fh:
    fh.write("quantity,value\n")
    for name, val in [("features", d_feat), ("source_tasks", len(source_cols)), ("wind_tasks", len(wind_cols)),
                      ("dprime", K), ("days_2006_2023", n_pre), ("days_2024", n_tgt), ("test_days_2024", int(test.sum())),
                      ("label_std_2024", round(float(y_tgt.std()), 3)), ("ceiling_rmse", round(ceiling, 3)),
                      ("persistence_rmse", round(persist, 3))]:
        fh.write(f"{name},{val}\n")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), gridspec_kw={"width_ratios": [3, 2]})
ax = axes[0]
styles = {"scratch": ("-", "o", "gray", "least squares from scratch, 960 features"),
          "ridge": ("--", "s", "gray", "ridge regression from scratch, 960 features"),
          "pretrained": ("-", "D", "black", "pretrained representation, 4 features"),
          "wind": ("-.", "^", "black", "pretrained on wind direction only, 4 features"),
          "mean": (":", None, "black", "mean of the labeled days")}
for k, (ls, mk, col, lab) in styles.items():
    ax.plot(SIZES, curve[k], ls, marker=mk, color=col, label=lab)
ax.axhline(ceiling, color="black", linestyle=(0, (1, 3)), linewidth=0.8, label="all 6209 labeled days of 2006-2023")
ax.set_xscale("log", base=2); ax.set_xticks(SIZES); ax.set_xticklabels(SIZES)
ax.set_ylim(0, 18)
ax.set_xlabel("number of labeled days of 2024\n(a)"); ax.set_ylabel("RMSE on the test days (deg C)")
ax.set_title("tomorrow's 15:00 temperature from five days of readings"); ax.legend(frameon=False, fontsize=7)
ax = axes[1]
ax.plot(RANKS, rank_curve, "-", marker="D", color="black")
ax.set_xscale("log", base=2); ax.set_xticks(RANKS); ax.set_xticklabels(RANKS)
ax.set_xlabel("dimension d' of the representation\n(b)"); ax.set_ylabel("RMSE on the test days (deg C)")
ax.set_title("16 labeled days, varying d'")
fig.tight_layout(); fig.savefig(OUT_DIR / "pretraining.png", dpi=110); plt.close(fig)
check("[B-plot] the CSV files and the preview were written",
      all((OUT_DIR / f).exists() for f in ("pretraining_curve.csv", "pretraining_rank.csv", "pretraining_data.csv", "pretraining.png")))
n_ok = sum(ok for _, ok in report)
print(f"\n{n_ok}/{len(report)} checks pass")